# 3.5 · 类别变量编码 / Categorical Encoding

> **课程定位 / Where this fits**
> **Part 3 第 5 课**。模型只吃数字, 但真实数据满是类别（性别/国家/职业/套餐）。怎么把它们变成数字**而不引入虚假的顺序、不爆维度、不泄漏目标**——这一课讲透。
> Models eat numbers, but real data is full of categories. Encoding without injecting fake order, exploding dimensions, or leaking the target.

> 💡 **面试相关 / Interview-relevant**
> - "label encoding 和 one-hot 区别, 什么时候用哪个" ★★★★★
> - "高基数类别（如邮编/用户ID）怎么编码" ★★★★（target/frequency/hashing）
> - "target encoding 的泄漏陷阱" ★★★★★（必须 K-fold 编码）
> - "树模型需要 one-hot 吗" ★★★

---

## 学习目标 / Learning Objectives
1. 区分**名义 vs 有序**类别, 用对编码（别给名义类别强加顺序）。
2. 掌握 5 种编码：Label / Ordinal / One-Hot / Target / Frequency, 各自适用与陷阱。
3. 处理**高基数**类别（一列几千个值）。
4. **Target encoding 的泄漏**——为什么必须 K-fold/留一编码。
5. 处理**测试集出现的新类别**（unseen categories）。

## 目录 / TOC
1. [名义 vs 有序 ⭐](#1)
2. [🧑 数据：模拟收入普查](#2)
3. [Label / Ordinal 编码](#3)
4. [One-Hot 编码 + 虚拟变量陷阱](#4)
5. [高基数问题](#5)
6. [Frequency 编码](#6)
7. [Target 编码 + 泄漏陷阱 ⭐](#7)
8. [处理未见类别](#8)
9. [选择决策表](#9)
10. [小结](#10)


<a id="1"></a>
## 1. 名义 vs 有序 ⭐ / Nominal vs Ordinal

**编码前必须分清的第一件事**：

| 类型 | 定义 | 例子 | 编码原则 |
|---|---|---|---|
| **名义 Nominal** | 无内在顺序 | 颜色、国家、性别 | **不能**强加数字顺序 → one-hot / target / freq |
| **有序 Ordinal** | 有内在顺序 | 学历(小学<中学<大学)、评分(差<中<好)、套餐(基础<高级<旗舰) | **可以**用保序整数 |

⚠ **最经典的错误**：把名义类别（如 country: US=0, UK=1, DE=2）当数字喂给模型 → 模型会**幻想出"DE > UK > US"的虚假大小关系**, 在线性模型/KNN 里灾难性。
The classic mistake: label-encoding a nominal feature makes the model hallucinate "DE > UK > US" ordering — disastrous for linear models and KNN.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 构造一个"收入预测"数据集 (adult-income 风格) / synthetic income dataset
n = 5000
education = rng.choice(["HS","Bachelor","Master","PhD"], n, p=[0.4,0.35,0.2,0.05])  # 有序!
country = rng.choice(["US","UK","DE","JP","CA","MX","BR","IN"], n)                   # 名义
occupation = rng.choice([f"job_{i}" for i in range(50)], n)                          # 高基数名义!

# 目标: 高收入 (受 education 影响, 模拟真实信号) / target depends on education
edu_effect = pd.Series({"HS":0.1,"Bachelor":0.3,"Master":0.5,"PhD":0.7})
p_high = edu_effect[education].values + rng.normal(0, 0.1, n)
high_income = (rng.random(n) < np.clip(p_high, 0, 1)).astype(int)

df = pd.DataFrame({"education":education,"country":country,"occupation":occupation,
                   "high_income":high_income})
print(f"shape: {df.shape}")
print(f"education 基数: {df.education.nunique()} (有序)")
print(f"country 基数: {df.country.nunique()} (名义)")
print(f"occupation 基数: {df.occupation.nunique()} (高基数名义!)")
print(f"目标 high_income 占比: {df.high_income.mean():.1%}")


<a id="3"></a>
## 3. Label / Ordinal 编码 / Label & Ordinal Encoding

| 编码器 | 用途 | 关键 |
|---|---|---|
| `LabelEncoder` | **只给目标 y** 编码 | 不该用于特征（按字母序乱编）|
| `OrdinalEncoder` | 有序特征, **手动指定顺序** | `categories=[[...]]` 保序 |

⚠ `OrdinalEncoder` 默认按字母序编码——对**有序**特征必须手动传 `categories` 指定真实顺序。


In [ ]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

# 有序特征: 手动指定顺序 / ordinal feature with explicit order
edu_order = [["HS","Bachelor","Master","PhD"]]
oe = OrdinalEncoder(categories=edu_order)
df["edu_ordinal"] = oe.fit_transform(df[["education"]]).astype(int)
print("education 有序编码 (保序):")
print(df.groupby("education")["edu_ordinal"].first().sort_values())

# ⚠ 默认 (字母序) 会乱: Bachelor=0, HS=1, Master=2, PhD=3 — HS 被排到 Bachelor 后面!
oe_wrong = OrdinalEncoder()
wrong = oe_wrong.fit_transform(df[["education"]])
print(f"\n默认字母序编码 (错!): {dict(zip(oe_wrong.categories_[0], range(4)))}")
print("→ HS(高中)=1 被排在 Bachelor=0 之后, 顺序全乱! 必须手动指定 categories")


<a id="4"></a>
## 4. One-Hot 编码 + 虚拟变量陷阱 / One-Hot & Dummy Trap

**名义类别的标准做法**：每个取值变成一个 0/1 列。无虚假顺序。

| 优点 | 缺点 |
|---|---|
| 无顺序假设, 通用 | 基数高时**维度爆炸** |
| 线性模型可解释 | 稀疏, 内存大 |

⚠ **虚拟变量陷阱 (dummy variable trap)**：$k$ 个类别只需 $k-1$ 列（最后一个由其余推出）。**线性回归必须 drop 一个**（否则完全共线, 矩阵不可逆）；**树模型/正则化模型不用 drop**。
For linear regression, k categories need only k-1 dummies (drop one to avoid perfect collinearity); trees and regularized models don't care.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

# country: 8 个名义类别 → one-hot / nominal -> one-hot
ohe = OneHotEncoder(sparse_output=False, drop=None)
country_oh = ohe.fit_transform(df[["country"]])
print(f"country (8类) → one-hot: {country_oh.shape[1]} 列")
print(f"列名: {ohe.get_feature_names_out()}")

# drop='first' 避免虚拟变量陷阱 (线性模型用) / drop one for linear models
ohe_drop = OneHotEncoder(sparse_output=False, drop="first")
print(f"\ndrop='first' → {ohe_drop.fit_transform(df[['country']]).shape[1]} 列 (k-1, 给线性回归)")

# pandas 快捷方式 / pandas shortcut
print(f"\npd.get_dummies 也行: {pd.get_dummies(df['country'], prefix='c').shape[1]} 列")


<a id="5"></a>
## 5. 高基数问题 / High Cardinality

`occupation` 有 50 个值；真实场景里**邮编(4万)/用户ID(百万)/商品SKU**更夸张。One-hot 会产生海量稀疏列：


In [ ]:
print(f"occupation one-hot → {df.occupation.nunique()} 列 (还能忍)")
print("但想象: 邮编 → 4 万列, 用户ID → 百万列 — one-hot 直接崩溃\n")
print("高基数的解法 (本课后面讲):")
print("  1. Frequency 编码 — 用出现频率代替 (1列, 第6节)")
print("  2. Target 编码 — 用该类别的目标均值代替 (1列, 第7节, 但有泄漏陷阱)")
print("  3. Hashing 编码 — 哈希到固定列数 (可控维度, 有碰撞)")
print("  4. Embedding — 神经网络学低维稠密向量 (Part 7/15, 推荐系统标配)")


In [ ]:
# Hashing 编码: 把任意基数压到固定列数 / hashing to fixed dims
from sklearn.feature_extraction import FeatureHasher
hasher = FeatureHasher(n_features=8, input_type="string")
hashed = hasher.transform([[v] for v in df["occupation"]]).toarray()
print(f"occupation (50类) → hashing → 固定 {hashed.shape[1]} 列")
print("优点: 维度可控, 流式数据/未见类别免疫; 缺点: 碰撞(不同类别撞同列), 不可解释")


<a id="6"></a>
## 6. Frequency 编码 / Frequency Encoding

**把每个类别替换成它的出现频率**。1 列搞定任意基数, 简单有效。

**直觉**：类别的"常见程度"本身可能有信息（罕见职业 vs 常见职业）。**缺点**：频率相同的不同类别会撞值。


In [ ]:
# frequency 编码: 类别 → 出现频率 / category -> its frequency
freq_map = df["occupation"].value_counts(normalize=True)
df["occ_freq"] = df["occupation"].map(freq_map)
print("occupation frequency 编码示例:")
print(df[["occupation","occ_freq"]].drop_duplicates().head(5).round(4))
print(f"\n50 个类别 → 1 列, 无维度爆炸")


<a id="7"></a>
## 7. Target 编码 + 泄漏陷阱 ⭐ / Target Encoding & Leakage

**最强的高基数编码**：把每个类别替换成**该类别下目标的均值**。
$$\text{encode}(c) = \frac{\sum_{i: x_i = c} y_i}{\#\{i: x_i = c\}}$$

直接利用了"类别和目标的关系", 信息量大、1 列搞定。**但藏着致命泄漏**：


In [ ]:
# ❌ 朴素 target 编码 (有泄漏!) / naive target encoding LEAKS
naive = df.groupby("occupation")["high_income"].mean()
df["occ_target_naive"] = df["occupation"].map(naive)
print("朴素 target 编码: 用全部数据(含该行自己)算类别均值")
print("问题: 编码 occupation X 时, X 类别的均值里包含了'这一行自己的 y'")
print("→ 标签泄漏进特征! 在 train 上看似完美, test 上崩盘 (尤其稀有类别)\n")

# 演示泄漏的严重性: 稀有类别 (只出现几次) 的编码几乎 = 自己的 y
rare = df["occupation"].value_counts()
rare_cat = rare.index[-1]
sub = df[df.occupation == rare_cat]
print(f"稀有类别 '{rare_cat}' 只出现 {len(sub)} 次")
print(f"它的朴素 target 编码 = {naive[rare_cat]:.2f}, 几乎直接复制了这几行的 y")


In [ ]:
# ✅ K-fold target 编码 (防泄漏) / K-fold target encoding, leak-free
from sklearn.model_selection import KFold

def kfold_target_encode(df, col, target, n_splits=5, smoothing=10, seed=0):
    global_mean = df[target].mean()
    encoded = np.zeros(len(df))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(df):
        # 用 train fold 算均值, 编码 val fold (val 行不参与自己的均值)
        tr, val = df.iloc[tr_idx], df.iloc[val_idx]
        stats = tr.groupby(col)[target].agg(["mean","count"])
        # 平滑: 稀有类别向全局均值收缩 (贝叶斯思想, 2.10!) / smoothing shrinks rare cats
        smooth = (stats["mean"]*stats["count"] + global_mean*smoothing) / (stats["count"]+smoothing)
        encoded[val_idx] = val[col].map(smooth).fillna(global_mean).values
    return encoded

df["occ_target_kfold"] = kfold_target_encode(df, "occupation", "high_income")
print("K-fold target 编码: 每行的编码只用'其他折'的数据算, 不含自己 → 无泄漏")
print(f"\n对比 (同一稀有类别):")
print(f"  朴素编码 (泄漏): {df[df.occupation==rare_cat]['occ_target_naive'].iloc[0]:.3f}")
print(f"  K-fold (安全): {df[df.occupation==rare_cat]['occ_target_kfold'].mean():.3f} (向全局均值 {df.high_income.mean():.3f} 收缩)")
print("\n💡 sklearn 1.3+ 内置 TargetEncoder, 自动 K-fold + 平滑, 优先用它")


**Target 编码的两个救命要素**：
1. **K-fold 编码**：编码某行时只用**其他折**的数据算均值 → 该行的 y 不进自己的编码
2. **平滑 (smoothing)**：稀有类别（样本少, 均值不可信）向全局均值**收缩**——这正是 2.10 贝叶斯先验的思想（伪计数）!

> 💡 **面试金句**："Target encoding without out-of-fold computation leaks the label — it's the single most common cause of train-test performance gaps that vanish in production."


<a id="8"></a>
## 8. 处理未见类别 / Unseen Categories

**生产环境必然遇到**：test/线上出现 train 里没有的类别（新国家、新商品）。各编码器的应对：

| 编码器 | 未见类别处理 |
|---|---|
| OneHotEncoder | `handle_unknown="ignore"` → 全 0 行 |
| OrdinalEncoder | `handle_unknown="use_encoded_value", unknown_value=-1` |
| Frequency | 映射缺失 → 填 0 或全局频率 |
| Target | 填全局均值（平滑的自然产物）|
| Hashing | **天然免疫**（任意字符串都能哈希）⭐ |


In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# train 只见过 4 类, test 出现新类 'BR_NEW' / unseen category at test time
train_country = pd.DataFrame({"country": ["US","UK","DE","JP"]*10})
test_country = pd.DataFrame({"country": ["US","BR_NEW","DE"]})

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(train_country)
test_oh = ohe.transform(test_country)
print(f"train 类别: {list(ohe.categories_[0])}")
print(f"test 'BR_NEW' (未见) 的 one-hot 行: {test_oh[1]} (全 0, 不报错)")
print("\n⚠ 默认 handle_unknown='error' 会在 test 直接崩 — 生产代码必须设 'ignore'")


<a id="9"></a>
## 9. 选择决策表 / Decision Table

```
先分: 名义 还是 有序?
  有序 → OrdinalEncoder (手动传 categories 保序)
  名义 → 看基数:
    低基数 (<10-15): One-Hot (线性模型 drop='first')
    中基数 (15-50):  One-Hot 还能忍, 或 Target/Frequency
    高基数 (>50):
       有目标且想要信息量 → Target 编码 (必须 K-fold + 平滑!) ⭐
       只要简单         → Frequency 编码
       流式/未见类别多   → Hashing
       深度学习         → Embedding (Part 7/15)
树模型: One-Hot 会稀释切分 → 高基数优先 Target/Ordinal
永远: handle_unknown 设好; 编码器只 fit train
```

> 💡 **树模型的特殊性**：one-hot 把一个高基数列拆成几十个稀疏列, 每个切分只能看一个类别, 削弱树的能力。**树模型对高基数类别优先用 Target/Ordinal 编码**（LightGBM/CatBoost 甚至原生支持类别列, Part 5）。
> For trees, one-hot dilutes splits — prefer target/ordinal for high cardinality (LightGBM/CatBoost handle categoricals natively).


<a id="10"></a>
## 10. 小结 / Summary

```
名义 vs 有序: 名义别强加顺序!
编码谱:
  Ordinal (有序, 手动保序)
  One-Hot (名义低基数; 线性模型 drop 一列防共线)
  Frequency (高基数, 1列, 简单)
  Target (高基数最强; 必须 K-fold + 平滑 防泄漏) ⭐
  Hashing (固定维度, 未见类别免疫, 有碰撞)
  Embedding (深度学习)
未见类别: handle_unknown 必设
防泄漏: 编码器只 fit train; target 编码用 out-of-fold
```

### 💡 面试速查
1. **名义别 label encode**（虚假顺序）
2. **高基数**：target(K-fold!) / frequency / hashing / embedding
3. **Target 编码必须 out-of-fold + 平滑**——否则泄漏标签
4. **平滑 = 贝叶斯先验**（稀有类别向全局均值收缩）
5. **树模型对高基数**优先 target/ordinal, 别 one-hot

### 下一节
**3.6 特征工程**——编码完成, 进入"创造新特征"的艺术：交互、多项式、分箱、日期、地理特征。
